In [ ]:
%%capture
import os
from pathlib import Path

import pandas as pd
from dj_notebook import activate

env_file = os.environ["META_ENV"]
reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
import_folder = reports_folder.parent / "import"
plus = activate(dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)

In [ ]:
import re
from clinicedc_constants import OTHER
from django.conf import settings
# tokens

IMP_TOKEN = re.compile(r"""
      (?<![a-z])[ij]mp(?![a-z])                # IMP, Imp, imp_complaints, JMP typo
    | toleratE?
    | side[\s_-]*effect
    | pill[\s_-]*burden
    | \b(?:drug|pill|tablet|medic\w*)s?\b[^;,]{0,30}?\b(?:sick|unwell|bad|ill)\b
    | \b(?:sick|unwell|bad|ill)\b[^;,]{0,30}?\b(?:drug|pill|tablet|medic)
""", re.I | re.X)

WITHDRAWAL_TOKEN = re.compile(r"""
      (?<!re)(?<!re-)(?<!re\s)consen           # consent, consen, consentient; not reconsent / re-consent
    | withdr[ae]w                              # withdraw, withdrawl, withdrawal, withdrawn, withdrew
    | \bopt(?:ed|s|ing)?[\s_-]*out
    | \bnot\s+willing
    | \bunwilling
    | \bnot\s+ready\s+to\s+continue
    | \b(?:don\W{0,2}t|do\s+not|doesn\W{0,2}t|does\s+not)\s+want\s+to\s+continue
""", re.I | re.X)

# OTHER_TOKEN = re.compile(rf"(?<![^,;\s]){re.escape(OTHER)}(?![^,;\s])")

OTHER_TEXTS = [OTHER, "Other reason (specify below)"]

OTHER_TOKEN = re.compile(
    r"(?<![^,;\s])(?:"
    + "|".join(re.escape(t) for t in sorted(OTHER_TEXTS, key=len, reverse=True))
    + r")(?![^,;\s])"
)

tz = settings.TIME_ZONE  # or the site's zone, e.g. "Africa/Dar_es_Salaam"

def to_local(s):
    s = pd.to_datetime(s)
    if s.dt.tz is None:          # naive values from Django are UTC
        s = s.dt.tz_localize("UTC")
    return s.dt.tz_convert(tz)


def replace_other(reasons, other_text):
    if pd.isna(reasons) or pd.isna(other_text) or not str(other_text).strip():
        return reasons
    text = str(other_text).strip()
    return OTHER_TOKEN.sub(lambda m: text, str(reasons))

In [ ]:
df_katie = pd.read_excel(analysis_folder / "kbates_endpoints_010926.xls")

df_erik = pd.read_csv(analysis_folder / "endpoints_202609022359.csv")


In [ ]:
df_katie

In [ ]:
df_katie[df_katie["subject_identifier"].isin(df_erik["subject_identifier"])]

In [ ]:
df_10 = df_erik[~df_erik["subject_identifier"].isin(df_katie["subject_identifier"])].reset_index(drop=True)


In [ ]:
from meta_prn.models import EndOfStudy
from edc_model_to_dataframe.read_frame_edc import read_frame_edc
# "105-60-0056-3": Not included as an endpoint because did not meet threshold for endpoint

# EoS dataframe
df_eos = read_frame_edc(
    EndOfStudy.objects.all(),
    drop_sys_columns=True,
    drop_action_item_columns=True,
    read_frame_verbose=True
).rename(columns={"offstudy_reason_id": "offstudy_reason", "other_offstudy_reason": "offstudy_reason_other"})

df_eos = df_eos[["subject_identifier", "offstudy_datetime", "offstudy_reason", "offstudy_reason_other", "comment"]].reset_index(drop=True)

df_eos["offstudy_reason"] = [
    replace_other(r, o)
    for r, o in zip(df_eos["offstudy_reason"], df_eos["offstudy_reason_other"])
]


In [ ]:
df_10 = df_10.merge(df_eos, on="subject_identifier", how="left")

In [ ]:
df_10

In [ ]:
# "105-30-0317-2": "diagnosed by clinician -  does not qualify by study endpoint"
# "105-40-0030-0": "mistakenly removed"